# 05 — Model 2: PSM ATT (Digital Twins)

This notebook estimates the Average Treatment Effect on the Treated (ATT) of booster adoption on churn using 1:1 nearest-neighbor propensity score matching with a caliper.

Diagnostic checks are moved to a separate notebook: `3.3_psm_diagnostics.ipynb`.

## Method Overview With Formulas

Let $T_i \in \{0,1\}$ indicate booster adoption, $Y_i \in \{0,1\}$ indicate churn, and $X_i$ be observed covariates.

### 1) Propensity score estimation

We estimate treatment selection with logistic regression:
$$
e(X_i) = P(T_i=1\mid X_i) = \Lambda(X_i^\top\beta), \quad \Lambda(z)=\frac{1}{1+e^{-z}}
$$

### 2) Nearest-neighbor matching in score space

For each treated unit $i$ ($T_i=1$), find control unit(s) $j$ ($T_j=0$) that minimize
$$
d(i,j)=|e(X_i)-e(X_j)|
$$
subject to caliper constraint $d(i,j) \le c$. This notebook uses $k=1$ match per treated unit and caliper $c=0.02$.

### 3) Counterfactual construction and ATT

With matched control set $\mathcal{M}(i)$ for treated unit $i$, estimate its counterfactual outcome as
$$
\widehat{Y_i(0)} = \frac{1}{|\mathcal{M}(i)|}\sum_{j\in\mathcal{M}(i)} Y_j
$$
Then estimate ATT over matched treated units $\mathcal{T}_m$:
$$
\widehat{\mathrm{ATT}} = \frac{1}{|\mathcal{T}_m|}\sum_{i\in\mathcal{T}_m}\left(Y_i-\widehat{Y_i(0)}\right)
$$

### 4) Interpretation

Because $Y=1$ means churn, a negative ATT means boosters reduce churn for users who actually adopted boosters.

In [ ]:
import numpy as np
import pandas as pd

try:
    from helpers import load_analysis_data, prepare_model_data
    from psm_helpers import fit_propensity, run_matching
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    sys.path.append(str((Path.cwd() / "notebooks").resolve()))
    from helpers import load_analysis_data, prepare_model_data
    from psm_helpers import fit_propensity, run_matching

# Core PSM specification for ATT
NUMERIC_COVARIATES = ["age", "tenure"]
CATEGORICAL_COVARIATES = ["internet_usage", "tv_product", "mobile_product", "commune"]
PS_FORMULA = (
    "has_booster ~ age + tenure + C(internet_usage) + "
    "C(tv_product) + C(mobile_product) + C(commune)"
)
BASE_CALIPER = 0.02
BASE_RATIO = 1

df = load_analysis_data()
df_clean = prepare_model_data(df).copy()
df_clean = df_clean.reset_index(drop=False).rename(columns={"index": "_orig_index"})
print(f"Rows for PSM ATT estimation: {len(df_clean):,}")

Rows for PSM diagnostics: 125,000


In [ ]:
# Estimate propensity scores and run baseline 1:1 matching
ps_model, ps_scores = fit_propensity(df_clean, formula=PS_FORMULA)
df_clean["propensity_score"] = ps_scores

result = run_matching(df_clean, caliper=BASE_CALIPER, ratio=BASE_RATIO, strata_col=None)
pairs = result["pairs"]

att_pp = result["att"] * 100
print(f"PSM ATT (caliper={BASE_CALIPER}, ratio={BASE_RATIO}:1): {att_pp:+.2f} pp churn")
print(f"Matched pairs: {result['n_pairs']:,}")
print(f"Matched treated: {result['matched_treated']:,}")
print(f"Treated coverage: {result['coverage']:.1%}")
print(f"Average match distance: {result['avg_distance']:.4f}")

att_summary = pd.DataFrame([{
    "att": result["att"],
    "att_pp": att_pp,
    "n_pairs": result["n_pairs"],
    "matched_treated": result["matched_treated"],
    "treated_coverage": result["coverage"],
    "avg_match_distance": result["avg_distance"],
}])
display(att_summary)

tier_att = (
    pairs.assign(effect=lambda d: d["treated_outcome"] - d["control_outcome"])
         .merge(df_clean[["_orig_index", "internet_usage"]], left_on="treated_orig", right_on="_orig_index", how="left")
         .groupby("internet_usage", observed=True)["effect"]
         .agg(att="mean", matched_pairs="count")
         .reset_index()
         .assign(att_pp=lambda d: d["att"] * 100)
)
display(tier_att)

Baseline ATT (caliper=0.02, ratio=1:1): -1.63 pp
Matched pairs: 25,000 | Coverage: 100.0%
Average match distance: 0.0000


## Identification Assumptions

The ATT interpretation relies on the standard matching assumptions:

1. Conditional independence (selection on observables):
$$
(Y(1),Y(0)) \perp T \mid X
$$

2. Overlap for treated units:
$$
0 < P(T=1\mid X) < 1
$$

3. Stable unit treatment value (no interference and well-defined treatment).

Under these assumptions, matching treated users to observationally similar control users identifies ATT for the treated population.

## Practical Notes

1. This estimate is causal only to the extent that all major confounders are included in $X$.
2. Caliper matching trades off bias and sample coverage: tighter calipers improve similarity but can drop treated units.
3. This ATT is local to matched treated users, not necessarily the full customer base.
4. Full diagnostics (balance, overlap, sensitivity, calibration, and robustness) are in `3.3_psm_diagnostics.ipynb`.